<a href="https://colab.research.google.com/github/Janpu-Hou/Green-Learning-Basic/blob/main/iot23_light_gbm%20v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Loading and Preprocessing the IoT-23 Dataset

This section outlines the general steps for loading and preprocessing a real IoT-23 dataset. You will need to adapt this code based on the actual format (e.g., CSV, JSON, Zeek logs), features, and target variable of your specific IoT-23 data. Common preprocessing steps include handling missing values, encoding categorical features, scaling numerical features, and feature engineering.

In [1]:
import os

# Download the dataset
!wget https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz

# Extract the dataset
!tar -xzvf iot_23_datasets_small.tar.gz

print("Download and extraction complete.")

--2026-07-28 20:39:32--  https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz
Resolving mcfp.felk.cvut.cz (mcfp.felk.cvut.cz)... 147.32.82.194
Connecting to mcfp.felk.cvut.cz (mcfp.felk.cvut.cz)|147.32.82.194|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9373916249 (8.7G) [application/x-gzip]
Saving to: ‘iot_23_datasets_small.tar.gz’

iot_23_datasets_sma 100%[===================>]   8.73G  14.9MB/s    in 12m 11s 

2026-07-28 20:51:44 (12.2 MB/s) - ‘iot_23_datasets_small.tar.gz’ saved [9373916249/9373916249]

opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-1-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-17-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-20-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-21-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT

In [2]:
# List the extracted labeled files to identify the correct paths
!find opt/Malware-Project -name "*.labeled"

opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-20-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-35-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-36-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-7-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-17-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-52-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-60-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-3-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-49-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-9-1/bro/conn.log.labeled
opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Cap

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def load_iot_data(file_path, nrows=100000):
    """Loads a sample of IoT-23 conn.log.labeled data into a pandas DataFrame."""
    column_names = [
        'ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
        'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
        'conn_state', 'local_orig', 'local_resp', 'missed_bytes',
        'history', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'label'
    ]

    try:
        df = pd.read_table(file_path, sep='\t', comment='#', header=None, low_memory=False, nrows=nrows)
        df.columns = column_names

        # Convert '-' to NaN for numeric columns
        numeric_cols = ['duration', 'orig_bytes', 'resp_bytes', 'id.orig_p', 'id.resp_p']
        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col].replace('-', np.nan), errors='coerce')

        # Fill NaN values with 0 after conversion, or choose a different strategy as appropriate
        df = df.fillna(0)

        print(f"Successfully loaded {len(df)} rows from {file_path}")
        return df
    except Exception as e:
        print(f"Error loading data from {file_path}: {e}")
        return pd.DataFrame()

# Use one of the extracted files for demonstration
sample_file = 'opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-1-1/bro/conn.log.labeled'
df_sample = load_iot_data(sample_file)
display(df_sample.head())

Successfully loaded 100000 rows from opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-1-1/bro/conn.log.labeled


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,label
0,1.525880e+09,CUmrqr4svHuSXJy5z7,192.168.100.103,51524,65.127.233.163,23,tcp,-,2.999051,0.0,...,S0,-,-,0,S,3,180,0,0,(empty) Malicious PartOfAHorizontalPortScan
1,1.525880e+09,CH98aB3s1kJeq6SFOc,192.168.100.103,56305,63.150.16.171,23,tcp,-,0.000000,0.0,...,S0,-,-,0,S,1,60,0,0,(empty) Malicious PartOfAHorizontalPortScan
2,1.525880e+09,C3GBTkINvXNjVGtN5,192.168.100.103,41101,111.40.23.49,23,tcp,-,0.000000,0.0,...,S0,-,-,0,S,1,60,0,0,(empty) Malicious PartOfAHorizontalPortScan
3,1.525880e+09,CDe43c1PtgynajGI6,192.168.100.103,60905,131.174.215.147,23,tcp,-,2.998796,0.0,...,S0,-,-,0,S,3,180,0,0,(empty) Malicious PartOfAHorizontalPortScan
4,1.525880e+09,CJaDcG3MZzvf1YVYI4,192.168.100.103,44301,91.42.47.63,23,tcp,-,0.000000,0.0,...,S0,-,-,0,S,1,60,0,0,(empty) Malicious PartOfAHorizontalPortScan


In [5]:
def map_to_detailed_label(label):
    label = str(label)
    if 'Benign' in label: return 'Benign'
    elif 'DDoS' in label: return 'DDoS'
    elif 'PartOfAHorizontalPortScan' in label: return 'PortScan'
    elif 'C&C' in label: return 'C&C'
    elif 'Attack' in label: return 'Attack'
    elif 'FileDownload' in label: return 'FileDownload'
    elif 'Malicious' in label: # Catch-all for other malicious types not explicitly listed above
        # Attempt to extract a more specific malicious type if present, otherwise default to 'Other Malicious'
        parts = label.split('   ')
        for part in parts:
            if part not in ['(empty)', 'Malicious', '-'] and part.strip() != '':
                return part.strip() # Return the specific malicious type
        return 'Other Malicious'
    else: return 'Other'

df_sample['detailed_label'] = df_sample['label'].apply(map_to_detailed_label)

print("Distribution of Detailed Labels:")
display(df_sample['detailed_label'].value_counts())

Distribution of Detailed Labels:


,count
detailed_label,
PortScan,55045
Benign,44955


In [8]:
import pandas as pd
import numpy as np
from collections import Counter

# Assuming 'files' variable is available from previous !find command
# If not, uncomment and run: files = !find opt/Malware-Project -name "*.labeled"

def map_to_detailed_label(label):
    label = str(label)
    if 'Benign' in label: return 'Benign'
    elif 'DDoS' in label: return 'DDoS'
    elif 'PartOfAHorizontalPortScan' in label: return 'PortScan'
    elif 'C&C' in label: return 'C&C'
    elif 'Attack' in label: return 'Attack'
    elif 'FileDownload' in label: return 'FileDownload'
    elif 'Malicious' in label:
        parts = label.split('   ')
        for part in parts:
            if part not in ['(empty)', 'Malicious', '-'] and part.strip() != '':
                return part.strip()
        return 'Other Malicious'
    else: return 'Other'

def get_all_detailed_labels(file_paths, chunksize=100000):
    all_labels = []
    for file_path in file_paths:
        print(f"Processing {file_path}...")
        try:
            for chunk in pd.read_table(file_path, sep='\t', comment='#', header=None, low_memory=False, chunksize=chunksize):
                chunk.columns = [
                    'ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
                    'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
                    'conn_state', 'local_orig', 'local_resp', 'missed_bytes',
                    'history', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'label'
                ]
                all_labels.extend(chunk['label'].apply(map_to_detailed_label).tolist())
        except Exception as e:
            print(f"Error reading {file_path}: {e}. Skipping.")
    return all_labels

# Make sure the 'files' variable is correctly defined, if not, define it.
if 'files' not in locals():
    files = !find opt/Malware-Project -name "*.labeled"

# Get all detailed labels from the entire dataset
full_dataset_detailed_labels = get_all_detailed_labels(files)

# Display the distribution of detailed labels for the entire dataset
label_counts = pd.Series(full_dataset_detailed_labels).value_counts()
print("\nFull Dataset Detailed Label Distribution:")
display(label_counts)

Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-20-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-35-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-36-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-7-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-17-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-52-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-60-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-3-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-49-1/bro/conn.log.labeled...
Processing opt/Malware-Project

,count
PortScan,213853817
Okiru,60990708
Benign,30858735
DDoS,19538713
C&C,56598
Attack,9401
Other,1956
FileDownload,18


### Creating a Balanced Dataset for LightGBM

To prepare the data for training a LightGBM classifier, we need a balanced dataset with a sufficient number of samples for each `detailed_label` category. We will sample 5000 instances of each unique malicious traffic type (`detailed_label`), along with 5000 benign samples, from the entire IoT-23 dataset. This ensures that the model is not biased towards over-represented classes and has enough data to learn the characteristics of all relevant attack types.

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Re-define map_to_detailed_label for clarity and self-containment
def map_to_detailed_label(label):
    label = str(label)
    if 'Benign' in label: return 'Benign'
    elif 'DDoS' in label: return 'DDoS'
    elif 'PartOfAHorizontalPortScan' in label: return 'PortScan'
    elif 'C&C' in label: return 'C&C'
    elif 'Attack' in label: return 'Attack'
    elif 'FileDownload' in label: return 'FileDownload'
    elif 'Malicious' in label:
        parts = label.split('   ')
        for part in parts:
            if part not in ['(empty)', 'Malicious', '-'] and part.strip() != '':
                return part.strip()
        return 'Other Malicious'
    else: return 'Other'

def load_and_sample_balanced_data(file_paths, samples_per_label=5000, chunksize=100000):
    # Initialize a dictionary to hold sampled dataframes for each label
    sampled_dfs = {label: [] for label in [
        'Benign', 'DDoS', 'PortScan', 'C&C', 'Attack', 'FileDownload', 'Okiru', 'Other Malicious', 'Other'
    ]}
    current_sample_counts = {label: 0 for label in sampled_dfs.keys()}

    # Define the columns we are interested in for features + label
    column_names = [
        'ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
        'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes',
        'conn_state', 'local_orig', 'local_resp', 'missed_bytes',
        'history', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'label'
    ]

    print(f"Starting to sample {samples_per_label} instances per detailed label...")

    for file_path in file_paths:
        print(f"Processing {file_path}...")
        try:
            for chunk in pd.read_table(file_path, sep='\t', comment='#', header=None, low_memory=False, chunksize=chunksize):
                chunk.columns = column_names

                # Apply detailed label mapping
                chunk['detailed_label'] = chunk['label'].apply(map_to_detailed_label)

                # Preprocess numeric columns for potential features
                numeric_cols = ['duration', 'orig_bytes', 'resp_bytes', 'id.orig_p', 'id.resp_p', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes']
                for col in numeric_cols:
                    chunk[col] = pd.to_numeric(chunk[col].replace('-', np.nan), errors='coerce').fillna(0)

                # Sample from each label within this chunk
                for label_type in sampled_dfs.keys():
                    if current_sample_counts[label_type] < samples_per_label:
                        sub_chunk = chunk[chunk['detailed_label'] == label_type]
                        remaining_needed = samples_per_label - current_sample_counts[label_type]

                        if len(sub_chunk) > 0:
                            # Sample without replacement if possible, otherwise take all
                            sample_size = min(len(sub_chunk), remaining_needed)
                            sampled_rows = sub_chunk.sample(n=sample_size, random_state=42)
                            sampled_dfs[label_type].append(sampled_rows)
                            current_sample_counts[label_type] += sample_size

                # Check if all labels have reached their target
                if all(count >= samples_per_label for count in current_sample_counts.values()):
                    print("Target samples collected for all labels.")
                    break # Stop processing chunks from this file

            if all(count >= samples_per_label for count in current_sample_counts.values()):
                break # Stop processing files

        except Exception as e:
            print(f"Error processing {file_path}: {e}. Skipping.")

    # Concatenate all collected samples
    final_balanced_df = pd.concat([pd.concat(dfs) for dfs in sampled_dfs.values() if dfs])

    print("\nFinal Balanced Dataset Distribution:")
    print(final_balanced_df['detailed_label'].value_counts())

    return final_balanced_df

# Ensure 'files' variable is available
if 'files' not in locals():
    files = !find opt/Malware-Project -name "*.labeled"

balanced_df = load_and_sample_balanced_data(files, samples_per_label=5000)

# Define features and target
selected_features = [
    'id.orig_p', 'id.resp_p', 'proto', 'service', 'duration',
    'orig_bytes', 'resp_bytes', 'conn_state', 'orig_pkts', 'orig_ip_bytes',
    'resp_pkts', 'resp_ip_bytes'
]
target_label = 'detailed_label'

X = balanced_df[selected_features].copy()
y = balanced_df[target_label].copy()

# Encode categorical features
categorical_features = ['proto', 'service', 'conn_state']
for col in categorical_features:
    if col in X.columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nBalanced Dataset created and split into training and testing sets.")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

print("\ny_train distribution:")
display(y_train.value_counts())
print("\ny_test distribution:")
display(y_test.value_counts())

Starting to sample 5000 instances per detailed label...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-20-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-35-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-36-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-7-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-17-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-52-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-60-1/bro/conn.log.labeled...


/tmp/ipykernel_1639/1499953925.py:52: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  chunk[col] = pd.to_numeric(chunk[col].replace('-', np.nan), errors='coerce').fillna(0)
/tmp/ipykernel_1639/1499953925.py:52: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  chunk[col] = pd.to_numeric(chunk[col].replace('-', np.nan), errors='coerce').fillna(0)
/tmp/ipykernel_1639/1499953925.py:52: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.

Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-3-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-49-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-9-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-42-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-44-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-8-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-Honeypot-Capture-5-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-39-1/bro/conn.log.labeled...


/tmp/ipykernel_1639/1499953925.py:52: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  chunk[col] = pd.to_numeric(chunk[col].replace('-', np.nan), errors='coerce').fillna(0)
/tmp/ipykernel_1639/1499953925.py:52: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  chunk[col] = pd.to_numeric(chunk[col].replace('-', np.nan), errors='coerce').fillna(0)
/tmp/ipykernel_1639/1499953925.py:52: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.

Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-Honeypot-Capture-4-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-43-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-1-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-34-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-Honeypot-Capture-7-1/Somfy-01/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-21-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-33-1/bro/conn.log.labeled...
Processing opt/Malware-Project/BigDataset/IoTScenarios/CTU-IoT-Malware-Capture-48-1/bro/conn.log.labeled...

Final Balanced Dataset Distribution:
detailed_label
Benign          5000
DDoS            5000
PortScan        5000
C&C             5000

,count
detailed_label,
DDoS,4000
PortScan,4000
Attack,4000
Benign,4000
C&C,4000
Okiru,4000
Other,1565
FileDownload,14



y_test distribution:


,count
detailed_label,
C&C,1000
Okiru,1000
PortScan,1000
Benign,1000
Attack,1000
DDoS,1000
Other,391
FileDownload,4


In [10]:
import lightgbm as lgb
from sklearn.metrics import classification_report, accuracy_score

# Initialize the LightGBM Classifier
lgbm_clf = lgb.LGBMClassifier(random_state=42)

# Train the model
print("Training LightGBM model...")
lgbm_clf.fit(X_train, y_train)
print("LightGBM model training complete.")

Training LightGBM model...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003933 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1362
[LightGBM] [Info] Number of data points in the train set: 25579, number of used features: 12
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -7.510470
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -2.793886
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

In [11]:
# Make predictions on the test set
y_pred = lgbm_clf.predict(X_test)

# Evaluate the model
print("\nModel Evaluation:")
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Model Evaluation:
Accuracy: 0.6396

Classification Report:
              precision    recall  f1-score   support

      Attack       0.98      0.83      0.90      1000
      Benign       0.21      0.01      0.02      1000
         C&C       0.38      0.99      0.55      1000
        DDoS       0.84      0.09      0.16      1000
FileDownload       0.00      0.00      0.00         4
       Okiru       0.95      1.00      0.98      1000
       Other       0.26      0.44      0.33       391
    PortScan       0.93      1.00      0.96      1000

    accuracy                           0.64      6395
   macro avg       0.57      0.54      0.49      6395
weighted avg       0.69      0.64      0.58      6395



In [12]:
from sklearn.model_selection import RandomizedSearchCV

# Define the hyperparameter search space for LightGBM
param_dist = {
    'num_leaves': [20, 31, 40, 50, 60],
    'max_depth': [5, 7, 10, 12, -1], # -1 means no limit
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300, 500],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0, 0.1, 0.5, 1.0],
    'min_child_samples': [10, 20, 30, 50]
}

# Initialize LGBMClassifier
lgbm = lgb.LGBMClassifier(random_state=42, objective='multiclass', num_class=len(y_train.unique()))

# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=lgbm,
                                   param_distributions=param_dist,
                                   n_iter=50, # Number of parameter settings that are sampled
                                   scoring='f1_weighted', # Use f1_weighted to account for class imbalance
                                   cv=3, # 3-fold cross-validation
                                   verbose=2,
                                   random_state=42,
                                   n_jobs=-1) # Use all available cores

print("Starting RandomizedSearchCV for hyperparameter tuning...")
random_search.fit(X_train, y_train)
print("RandomizedSearchCV complete.")

# Print the best hyperparameters and the best score
print("\nBest Hyperparameters:", random_search.best_params_)
print("Best F1 Weighted Score:", random_search.best_score_)


Starting RandomizedSearchCV for hyperparameter tuning...
Fitting 3 folds for each of 50 candidates, totalling 150 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003838 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1362
[LightGBM] [Info] Number of data points in the train set: 25579, number of used features: 12
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -7.510470
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -2.793886
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

In [13]:
# Train a new model with the best hyperparameters
best_lgbm_clf = random_search.best_estimator_

print("\nTraining LightGBM model with best hyperparameters...")
best_lgbm_clf.fit(X_train, y_train)
print("LightGBM model with best hyperparameters training complete.")

# Make predictions on the test set with the best model
y_pred_tuned = best_lgbm_clf.predict(X_test)

# Evaluate the tuned model
print("\nModel Evaluation with Tuned Hyperparameters:")
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
print(f"Accuracy (Tuned Model): {accuracy_tuned:.4f}")

print("\nClassification Report (Tuned Model):")
print(classification_report(y_test, y_pred_tuned))


Training LightGBM model with best hyperparameters...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003838 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1362
[LightGBM] [Info] Number of data points in the train set: 25579, number of used features: 12
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -7.510470
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Info] Start training from score -2.793886
[LightGBM] [Info] Start training from score -1.855477
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


In [4]:
import joblib

# Define the filename for the saved model
model_filename = 'lgbm_tuned_model.joblib'

# Save the trained model to disk
joblib.dump(best_lgbm_clf, model_filename)
print(f"Tuned LightGBM model saved to {model_filename}")

NameError: name 'best_lgbm_clf' is not defined

In [ ]:
# Load the model from disk
loaded_model = joblib.load(model_filename)
print(f"Model loaded from {model_filename}")

# You can now use the loaded_model for predictions
# For example, to make predictions on X_test:
loaded_predictions = loaded_model.predict(X_test)

# And verify its performance (optional)
print("\nVerification of Loaded Model Performance:")
accuracy_loaded = accuracy_score(y_test, loaded_predictions)
print(f"Accuracy (Loaded Model): {accuracy_loaded:.4f}")
print("\nClassification Report (Loaded Model):")
print(classification_report(y_test, loaded_predictions))

You can load the saved model back into memory at any time using `joblib.load()`:

In [3]:
# Load the model from disk
loaded_model = joblib.load(model_filename)
print(f"Model loaded from {model_filename}")

# You can now use the loaded_model for predictions
# For example, to make predictions on X_test:
loaded_predictions = loaded_model.predict(X_test)

# And verify its performance (optional)
print("\nVerification of Loaded Model Performance:")
accuracy_loaded = accuracy_score(y_test, loaded_predictions)
print(f"Accuracy (Loaded Model): {accuracy_loaded:.4f}")
print("\nClassification Report (Loaded Model):")
print(classification_report(y_test, loaded_predictions))

NameError: name 'model_filename' is not defined

In [2]:
import joblib

# Define filenames for the datasets
x_train_filename = 'X_train.joblib'
x_test_filename = 'X_test.joblib'
y_train_filename = 'y_train.joblib'
y_test_filename = 'y_test.joblib'

# Save the datasets
joblib.dump(X_train, x_train_filename)
joblib.dump(X_test, x_test_filename)
joblib.dump(y_train, y_train_filename)
joblib.dump(y_test, y_test_filename)

print(f"X_train saved to {x_train_filename}")
print(f"X_test saved to {x_test_filename}")
print(f"y_train saved to {y_train_filename}")
print(f"y_test saved to {y_test_filename}")

NameError: name 'X_train' is not defined

You can load these datasets back into memory at any time using `joblib.load()`:

```python
# Example of loading the datasets back
loaded_X_train = joblib.load('X_train.joblib')
loaded_X_test = joblib.load('X_test.joblib')
loaded_y_train = joblib.load('y_train.joblib')
loaded_y_test = joblib.load('y_test.joblib')

print("Datasets loaded successfully!")
```

In [1]:
import joblib
# Example of loading the datasets back
loaded_X_train=joblib.load('X_train.joblib')
loaded_X_test=joblib.load('X_test.joblib')
loaded_y_train=joblib.load('y_train.joblib')
loaded_y_test=joblib.load('y_test.joblib')

print("Datasets loaded successfully!")

Datasets loaded successfully!


You can load these datasets back into memory at any time using `joblib.load()`:

```python
# Example of loading the datasets back
loaded_X_train = joblib.load('X_train.joblib')
loaded_X_test = joblib.load('X_test.joblib')
loaded_y_train = joblib.load('y_train.joblib')
loaded_y_test = joblib.load('y_test.joblib')

print("Datasets loaded successfully!")
```

In [7]:
# Load the model from disk
model_filename = 'lgbm_tuned_model.joblib'
loaded_model = joblib.load(model_filename)
print(f"Model loaded from {model_filename}")

# You can now use the loaded_model for predictions
# For example, to make predictions on X_test:
loaded_predictions = loaded_model.predict(X_test)

# And verify its performance (optional)
print("\nVerification of Loaded Model Performance:")
accuracy_loaded = accuracy_score(y_test, loaded_predictions)
print(f"Accuracy (Loaded Model): {accuracy_loaded:.4f}")
print("\nClassification Report (Loaded Model):")
print(classification_report(y_test, loaded_predictions))

Model loaded from lgbm_tuned_model.joblib


NameError: name 'X_test' is not defined

In [8]:
import joblib
# Example of loading the datasets back
X_train=joblib.load('X_train.joblib')
X_test=joblib.load('X_test.joblib')
y_train=joblib.load('y_train.joblib')
y_test=joblib.load('y_test.joblib')

print("Datasets loaded successfully!")

Datasets loaded successfully!


In [11]:
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Load the model from disk
model_filename = 'lgbm_tuned_model.joblib'
loaded_model = joblib.load(model_filename)
print(f"Model loaded from {model_filename}")

# You can now use the loaded_model for predictions
# For example, to make predictions on X_test:
loaded_predictions = loaded_model.predict(X_test)

# And verify its performance (optional)
print("\nVerification of Loaded Model Performance:")
accuracy_loaded = accuracy_score(y_test, loaded_predictions)
print(f"Accuracy (Loaded Model): {accuracy_loaded:.4f}")
print("\nClassification Report (Loaded Model):")
print(classification_report(y_test, loaded_predictions))

Model loaded from lgbm_tuned_model.joblib

Verification of Loaded Model Performance:
Accuracy (Loaded Model): 0.9989

Classification Report (Loaded Model):
              precision    recall  f1-score   support

      Attack       1.00      1.00      1.00      1000
      Benign       1.00      1.00      1.00      1000
         C&C       1.00      1.00      1.00      1000
        DDoS       1.00      1.00      1.00      1000
FileDownload       1.00      1.00      1.00         4
       Okiru       1.00      1.00      1.00      1000
       Other       0.99      1.00      1.00       391
    PortScan       1.00      1.00      1.00      1000

    accuracy                           1.00      6395
   macro avg       1.00      1.00      1.00      6395
weighted avg       1.00      1.00      1.00      6395



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils import resample


# ==========================================
# STAGE 1: Saab Subspace Feature Extraction
# ==========================================
def stage1_saab_transform(df_raw, n_components=16):
    exclude_cols = ['label', 'detailed-label', 'id.orig_h', 'id.resp_h']
    numeric_cols = [c for c in df_raw.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df_raw[c])]

    X_raw = df_raw[numeric_cols].fillna(0).values
    y = df_raw['label'].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)

    pca = PCA(n_components=n_components, random_state=42)
    X_saab = pca.fit_transform(X_scaled)

    saab_cols = [f'saab_feat_{i+1}' for i in range(n_components)]
    df_saab = pd.DataFrame(X_saab, columns=saab_cols)
    df_saab['label'] = y

    return df_saab, saab_cols


# ==========================================
# STAGE 2: Discriminant Feature Test (DFT)
# ==========================================
def compute_single_dft_loss(x, y, n_bins=16):
    if np.min(x) == np.max(x):
        return 1.0
    bins = np.linspace(np.min(x), np.max(x), n_bins + 1)
    bin_idx = np.digitize(x, bins[:-1])

    unique_labels, y_encoded = np.unique(y, return_inverse=True)
    N = len(x)
    min_entropy = float('inf')

    for b in range(1, n_bins):
        left_mask = bin_idx <= b
        right_mask = ~left_mask
        N_L, N_R = np.sum(left_mask), np.sum(right_mask)
        if N_L == 0 or N_R == 0:
            continue

        _, counts_L = np.unique(y_encoded[left_mask], return_counts=True)
        probs_L = counts_L / N_L
        H_L = -np.sum(probs_L * np.log2(probs_L + 1e-12))

        _, counts_R = np.unique(y_encoded[right_mask], return_counts=True)
        probs_R = counts_R / N_R
        H_R = -np.sum(probs_R * np.log2(probs_R + 1e-12))

        weighted_H = (N_L / N) * H_L + (N_R / N) * H_R
        if weighted_H < min_entropy:
            min_entropy = weighted_H

    return min_entropy

def stage2_dft_pruning(df_saab, saab_cols, top_k=10, n_bins=16):
    y = df_saab['label'].values
    dft_scores = {}

    for col in saab_cols:
        loss = compute_single_dft_loss(df_saab[col].values, y, n_bins=n_bins)
        dft_scores[col] = loss

    sorted_dft = sorted(dft_scores.items(), key=lambda x: x[1])
    selected_features = [f[0] for f in sorted_dft[:top_k]]

    df_pruned = df_saab[selected_features].copy()
    df_pruned.loc[:, 'label'] = y

    return df_pruned, sorted_dft, selected_features

def balance_feature_matrix(df, target_count_per_class=10000, noise_std=0.01, random_state=42):
    np.random.seed(random_state)
    feature_cols = [c for c in df.columns if c != 'label']
    balanced_chunks = []

    for label, group in df.groupby('label'):
        n_samples = len(group)
        if n_samples < target_count_per_class:
            n_needed = target_count_per_class - n_samples
            oversampled_boot = resample(group, replace=True, n_samples=n_needed, random_state=random_state)

            X_feats = oversampled_boot[feature_cols].values
            feature_stds = group[feature_cols].std().values
            feature_stds = np.where(feature_stds == 0, 1e-4, feature_stds)

            noise = np.random.normal(0, noise_std, size=X_feats.shape) * feature_stds
            X_jittered = X_feats + noise

            df_synthetic = pd.DataFrame(X_jittered, columns=feature_cols)
            df_synthetic.loc[:, 'label'] = label
            group_balanced = pd.concat([group, df_synthetic], axis=0)
        else:
            group_balanced = resample(group, replace=False, n_samples=target_count_per_class, random_state=random_state)

        balanced_chunks.append(group_balanced)

    return pd.concat(balanced_chunks, axis=0).sample(frac=1.0, random_state=random_state).reset_index(drop=True)


# ==========================================
# STAGE 3: Subspace Learning Machine (SLM)
# ==========================================
class SLMNode:
    def __init__(self, depth=0, max_depth=6, min_samples_split=10,
                 n_projections=30, active_features_range=(2, 5), n_bins=16):
        self.depth = depth
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_projections = n_projections
        self.active_features_range = active_features_range
        self.n_bins = n_bins

        self.is_leaf = False
        self.prediction = None
        self.left = None
        self.right = None

        self.projection_vector = None
        self.selected_indices = None
        self.threshold = None

    def _compute_entropy(self, y):
        if len(y) == 0:
            return 0.0
        _, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return -np.sum(probs * np.log2(probs + 1e-12))

    def _generate_dft_weighted_projection(self, num_features, dft_losses):
        k = np.random.randint(self.active_features_range[0],
                              min(self.active_features_range[1] + 1, num_features + 1))
        active_indices = np.random.choice(num_features, size=k, replace=False)

        raw_coefficients = np.random.uniform(-1.0, 1.0, size=k)
        selected_dft_losses = dft_losses[active_indices]
        inv_dft_weights = 1.0 / (selected_dft_losses + 1e-8)

        a_k = raw_coefficients * inv_dft_weights
        norm = np.linalg.norm(a_k)
        if norm > 0:
            a_k = a_k / norm

        return active_indices, a_k

    def fit(self, X, y, dft_losses):
        unique_classes, counts = np.unique(y, return_counts=True)
        self.prediction = unique_classes[np.argmax(counts)]

        if (self.depth >= self.max_depth or
            len(y) < self.min_samples_split or
            len(unique_classes) == 1):
            self.is_leaf = True
            return

        best_gain = -1.0
        best_indices, best_a_k, best_threshold = None, None, None
        current_entropy = self._compute_entropy(y)
        N = len(y)

        for _ in range(self.n_projections):
            indices, a_k = self._generate_dft_weighted_projection(X.shape[1], dft_losses)
            X_projected = np.dot(X[:, indices], a_k)

            bins = np.linspace(np.min(X_projected), np.max(X_projected), self.n_bins + 1)
            for t in bins[1:-1]:
                left_mask = X_projected <= t
                right_mask = ~left_mask

                N_L, N_R = np.sum(left_mask), np.sum(right_mask)
                if N_L == 0 or N_R == 0:
                    continue

                H_L = self._compute_entropy(y[left_mask])
                H_R = self._compute_entropy(y[right_mask])
                weighted_H = (N_L / N) * H_L + (N_R / N) * H_R
                info_gain = current_entropy - weighted_H

                if info_gain > best_gain:
                    best_gain = info_gain
                    best_indices, best_a_k, best_threshold = indices, a_k, t

        if best_gain <= 0 or best_indices is None:
            self.is_leaf = True
            return

        self.selected_indices, self.projection_vector, self.threshold = best_indices, best_a_k, best_threshold
        X_projected_best = np.dot(X[:, best_indices], best_a_k)
        left_mask = X_projected_best <= best_threshold
        right_mask = ~left_mask

        self.left = SLMNode(self.depth + 1, self.max_depth, self.min_samples_split,
                            self.n_projections, self.active_features_range, self.n_bins)
        self.right = SLMNode(self.depth + 1, self.max_depth, self.min_samples_split,
                             self.n_projections, self.active_features_range, self.n_bins)

        self.left.fit(X[left_mask], y[left_mask], dft_losses)
        self.right.fit(X[right_mask], y[right_mask], dft_losses)

    def predict_one(self, x):
        if self.is_leaf:
            return self.prediction

        val = np.dot(x[self.selected_indices], self.projection_vector)
        if val <= self.threshold:
            return self.left.predict_one(x)
        else:
            return self.right.predict_one(x)

class SubspaceLearningMachine:
    def __init__(self, max_depth=6, min_samples_split=10, n_projections=30, n_bins=16, random_state=42):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_projections = n_projections
        self.n_bins = n_bins
        self.random_state = random_state
        self.root = None

    def fit(self, X, y, dft_losses):
        np.random.seed(self.random_state)
        self.root = SLMNode(max_depth=self.max_depth, min_samples_split=self.min_samples_split,
                            n_projections=self.n_projections, n_bins=self.n_bins)
        self.root.fit(X, y, dft_losses)

    def predict(self, X):
        return np.array([self.root.predict_one(x) for x in X])


# ==========================================
# MAIN PIPELINE EXECUTION
# ==========================================
if __name__ == '__main__':
    # 1. Load dataset
    print("1. Load Dataset")
    df_raw = pd.read_csv('./iot23_preprocessed.csv')

    # 2. Stage 1: Saab Transform
    print("2. Saab Transform")
    df_saab, saab_cols = stage1_saab_transform(df_raw, n_components=16)

    # 3. Stage 2: DFT Pruning & Balancing
    print("3. DFT Pruning & Balancing")
    df_pruned, sorted_dft, selected_features = stage2_dft_pruning(df_saab, saab_cols, top_k=10)
    df_balanced = balance_feature_matrix(df_pruned, target_count_per_class=10000)

    # 4. Train/Test Split
    print("4. Train/Test Split")
    X_cols = selected_features
    X_bal = df_balanced[X_cols].values
    y_bal = df_balanced['label'].values
    dft_losses_bal = np.array([compute_single_dft_loss(df_balanced[col].values, y_bal) for col in X_cols])

    X_train, X_test, y_train, y_test = train_test_split(
        X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
    )

    # 5. Stage 3: SLM Fit & Predict
    print("5. SLM Fit & Predict")
    slm = SubspaceLearningMachine(max_depth=6, n_projections=30, random_state=42)
    slm.fit(X_train, y_train, dft_losses_bal)
    y_pred = slm.predict(X_test)

    print(f"Full Pipeline Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

    # 6. Plot & Save Confusion Matrix
    print("6. Plot & Save Confusion Matrix")
    class_names = sorted(np.unique(y_test))
    cm = confusion_matrix(y_test, y_pred, labels=class_names)

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Sample Count'}
    )

    plt.title('Stage 3 SLM Final Confusion Matrix (Balanced Dataset)', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Predicted Label', fontsize=12, labelpad=10)
    plt.ylabel('True Label', fontsize=12, labelpad=10)
    plt.tight_layout()

    # Save high-resolution plot
    plt.savefig('slm_confusion_matrix.png', dpi=300)
    print("Saved confusion matrix plot to 'slm_confusion_matrix.png'!")
    plt.show()